In [15]:
import numpy as np
import pandas as pd


In [16]:
import pickle

def load_model(model_path):
    with open(model_path, "rb") as f:
        obj = pickle.load(f)
    print("Model loaded!")
    return obj["model"], obj["feature_cols"]

model, feature_cols = load_model(r".\model\yodogawa_match_model.pkl")

Model loaded!


In [17]:
def load_model(model_path):
    """Load trained matching model + feature list"""
    try:
        with open(model_path, "rb") as f:
            obj = pickle.load(f)
        print("Model loaded successfully!")
        return obj["model"], obj["feature_cols"]
    except Exception as e:
        raise RuntimeError(f"❌ ERROR loading model: {str(e)}")


#2 PREPROCESS DATA

In [18]:
def preprocess_property_df(df):
    """Standardize schema + build features expected by the trained model."""

    # --------------------------
    # Fix Japanese/alternate -> English columns
    # --------------------------
    column_mapping = {
        "\u7ba1\u7406\u8cbb_\u5171\u76ca\u8cbb": "management_fee",
        "\u968e": "floor",
        "floor_raw": "floor",
        "\u5411\u304d": "direction",
        "\u6761\u4ef6": "conditions",
    }
    df = df.rename(columns={c: column_mapping[c] for c in column_mapping if c in df.columns}).copy()

    # --------------------------
    # Numeric clean-up helper
    # --------------------------
    def _to_num(series):
        return pd.to_numeric(
            series.astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("\u00a5", "", regex=False)
            .str.replace("\u5186", "", regex=False)
            .str.extract(r"([-]?\d+\.?\d*)", expand=False),
            errors="coerce",
        )

    # Core model features
    for col in ["rent", "management_fee", "area_m2", "building_age"]:
        if col in df.columns:
            df[col] = _to_num(df[col]).fillna(0)
        else:
            df[col] = 0

    # Floor
    if "floor" in df.columns:
        df["floor"] = pd.to_numeric(
            df["floor"].astype(str).str.extract(r"(\d+)", expand=False),
            errors="coerce",
        ).fillna(0)
    else:
        df["floor"] = 0

    # Walking distance (minutes)
    if "walking_distance_to_station" in df.columns:
        wd = df["walking_distance_to_station"].astype(str).str.extract(r"(\d+)", expand=False)
        df["walking_distance_to_station"] = pd.to_numeric(wd, errors="coerce").fillna(0)
    else:
        df["walking_distance_to_station"] = 0

    # Pet flag
    if "conditions" in df.columns:
        df["pet_allowed"] = df["conditions"].astype(str).str.contains("\u30da\u30c3\u30c8", regex=False).astype(int)
    else:
        df["pet_allowed"] = 0

    # South-facing flag
    if "direction" in df.columns:
        df["south_facing_flag"] = df["direction"].astype(str).str.contains("\u5357", regex=False).astype(int)
    else:
        df["south_facing_flag"] = 0

    # Derived features (aligned with training pipeline)
    rent_yen = df["rent"] * 10000
    total_cost = rent_yen + df["management_fee"]
    df["total_cost"] = total_cost

    total_cost_nonzero = total_cost.replace(0, np.nan)
    df["cost_performance"] = (df["area_m2"] / total_cost_nonzero).fillna(0)

    df["distance_score"] = 1 / (df["walking_distance_to_station"] + 1)
    area_nonzero = df["area_m2"].replace(0, np.nan)
    df["price_per_m2"] = (rent_yen / area_nonzero).fillna(0)

    return df


#3. CREATE USER VECTOR

In [19]:
def compute_user_similarity(row, user):
    """Calculate soft similarity score between user criteria & property"""

    score = 0

    # AREA
    if row["area_m2"] >= user["min_area"]:
        score += 1

    # RENT
    if row["total_cost"] <= user["max_rent"]:
        score += 1

    # AGE
    if row["building_age"] <= user["max_age"]:
        score += 1

    # DISTANCE
    if row["walking_distance_to_station"] <= user["max_distance"]:
        score += 1

    # DIRECTION
    if user["south_facing"] and row["south_facing_flag"] == 1:
        score += 1

    # PET
    if (user["allow_pets"] is False) or (row["pet_allowed"] == user["allow_pets"]):
        score += 1

    return score

#4. COMPUTE MATCHING SCORE

In [20]:
def recommend_top5(df, model, feature_cols, **user_inputs):
    """Generate top 5 recommended properties using ML + user constraints"""

    df = df.copy()

    # -----------------------------------------------------------
    # ML prediction
    # -----------------------------------------------------------
    X = df.reindex(columns=feature_cols, fill_value=0)
    if hasattr(model, "predict_proba"):
        df["ml_score"] = model.predict_proba(X)[:, 1]
    else:
        # Regressor fallback (e.g., XGBRegressor)
        df["ml_score"] = model.predict(X)

    # -----------------------------------------------------------
    # User similarity scoring
    # -----------------------------------------------------------
    df["similarity_score"] = df.apply(
        lambda row: compute_user_similarity(row, user_inputs), axis=1
    )

    # -----------------------------------------------------------
    # Final blended score
    # -----------------------------------------------------------
    df["final_score"] = (
        0.7 * df["ml_score"] +
        0.3 * (df["similarity_score"] / df["similarity_score"].max())
    )

    df = df.sort_values("final_score", ascending=False)

    # -----------------------------------------------------------
    # Return top 5
    # -----------------------------------------------------------
    top5 = df.head(5)

    explanation = (
        "Top 5 recommended properties ranked by: \n"
        "- ML predicted matching score (70%)\n"
        "- Similarity to your personal preferences (30%)\n"
        "The system applies a soft-ranking approach, meaning even if no property meets all criteria, "
        "the engine recommends the closest matching options.\n"
    )

    return top5, explanation

#5. FINAL PREDICTION ENGINE

In [21]:
if __name__ == "__main__":

    MODEL_PATH = r".\model\yodogawa_match_model.pkl"
    INPUT_CSV = r".\data\yodogawa_feature_eng.csv"

    # Load model
    model, feature_cols = load_model(MODEL_PATH)

    # Load property dataset
    df = pd.read_csv(INPUT_CSV)

    # Preprocess df
    df = preprocess_property_df(df)

    # Run recommendation
    top5, explanation = recommend_top5(
        df, model, feature_cols,
        income=300000,
        persons=2,
        max_rent=120000,
        max_distance=15,
        min_area=25,
        max_age=20,
        south_facing=True,
        allow_pets=False,
    )

    print("===== TOP 5 RESULTS =====")
    print(top5[["id", "rent", "area_m2", "total_cost", "final_score"]])

    print("\n===== EXPLANATION =====")
    print(explanation)

Model loaded successfully!
===== TOP 5 RESULTS =====
        id  rent  area_m2  total_cost  final_score
639    649   3.5    22.00     38000.0    62.686839
1331  1394   6.0    29.08     63400.0    59.457982
1332  1395   6.0    29.08     63700.0    59.157475
621    629   3.6    24.87     44000.0    59.156771
1334  1398   6.1    30.45     64800.0    58.931522

===== EXPLANATION =====
Top 5 recommended properties ranked by: 
- ML predicted matching score (70%)
- Similarity to your personal preferences (30%)
The system applies a soft-ranking approach, meaning even if no property meets all criteria, the engine recommends the closest matching options.

